# Reusable Sector Pipeline Parity Validation

This notebook validates that the reusable multi-sector pipeline reproduces the established XLK and XLE results without overwriting them. It reads local raw data only. The 0.05–0.20 range remains the primary moderate-noise evidence; 0.50–1.00 remains an exploratory bounded-input stress test for which clipping is reported explicitly.


In [ ]:
from pathlib import Path
import hashlib
import json
import re
import sys
import tempfile
import numpy as np
import pandas as pd

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'src').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise RuntimeError('Run this notebook from inside the cloned repository.')

ROOT = find_project_root()
sys.path.insert(0, str(ROOT / 'src'))
from sector_config import PREDICTORS
from sector_robustness_pipeline import (
    CLIPPING_KEYS, DETAIL_KEYS, SUMMARY_KEYS, create_cross_sector_comparison,
    create_validation_output, hash_files, load_local_ohlcv_csv, run_sector_pipeline,
)

protected = sorted(
    [p for p in (ROOT/'notebooks').glob('0[1-6]_*.ipynb')]
    + [p for base in (ROOT/'data', ROOT/'outputs') for p in base.rglob('*xlk*')]
    + [p for base in (ROOT/'data', ROOT/'outputs') for p in base.rglob('*xle*')]
)
protected = [p for p in protected if p.is_file()]
def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()
protected_before = {str(p.relative_to(ROOT)): sha(p) for p in protected}
temporary_root = Path(tempfile.mkdtemp(prefix='sector-pipeline-parity-'))
print(f'Project root confirmed: {ROOT}')
print(f'Temporary validation directory: {temporary_root}')


## Parity helpers

Tables are sorted with explicit stable keys. Text and assignments must match exactly. Numeric columns are compared with absolute and relative tolerance `1e-12`; failures identify the ticker, file, columns and maximum difference.


In [ ]:
def compare_frame(actual, expected, ticker, file_name, keys, ignore=()):
    actual = actual.drop(columns=list(ignore), errors='ignore').sort_values(list(keys), kind='stable').reset_index(drop=True)
    expected = expected.drop(columns=list(ignore), errors='ignore').sort_values(list(keys), kind='stable').reset_index(drop=True)
    if set(actual.columns) != set(expected.columns):
        raise AssertionError(f'{ticker} {file_name}: column mismatch; actual-only={set(actual)-set(expected)}, expected-only={set(expected)-set(actual)}')
    actual = actual[expected.columns]
    if len(actual) != len(expected):
        raise AssertionError(f'{ticker} {file_name}: row mismatch {len(actual)} versus {len(expected)}')
    numeric = expected.select_dtypes(include=np.number).columns.tolist()
    text = [c for c in expected.columns if c not in numeric]
    unequal_text = [c for c in text if not actual[c].fillna('<NA>').equals(expected[c].fillna('<NA>'))]
    differences = {}
    for column in numeric:
        left = actual[column].to_numpy(float); right = expected[column].to_numpy(float)
        finite = np.isfinite(left) & np.isfinite(right)
        differences[column] = float(np.max(np.abs(left[finite]-right[finite]))) if finite.any() else 0.0
        if not np.allclose(left, right, rtol=1e-12, atol=1e-12, equal_nan=True):
            raise AssertionError(f'{ticker} {file_name}: numeric mismatch in {column}; maximum difference={differences[column]:.17g}')
    if unequal_text:
        raise AssertionError(f'{ticker} {file_name}: text mismatch in columns {unequal_text}')
    return max(differences.values(), default=0.0), differences

def established_clean(ticker):
    if ticker == 'XLK':
        logistic = pd.read_csv(ROOT/'outputs/tables/xlk_logistic_baseline_metrics.csv').query("model == 'LogisticRegression'").copy()
        logistic['model'] = 'Logistic Regression'
        forest = pd.read_csv(ROOT/'outputs/tables/xlk_random_forest_test_metrics.csv').copy()
        return pd.concat([logistic, forest], ignore_index=True)[['model','accuracy','balanced_accuracy','precision','recall','f1_score','roc_auc','average_precision']]
    return pd.read_csv(ROOT/'outputs/tables/xle_clean_baseline_metrics.csv')

def established_threshold(ticker):
    if ticker == 'XLK':
        table = pd.read_csv(ROOT/'outputs/tables/xlk_feasibility_summary.csv')
        return float(table.loc[table.metric.eq('training_75th_percentile_threshold'),'value'].iloc[0])
    notebook = json.loads((ROOT/'notebooks/06_xle_cross_sector_replication.ipynb').read_text())
    text = '\n'.join(''.join(output.get('text', [])) for cell in notebook['cells'] for output in cell.get('outputs', []))
    match = re.search(r'threshold ([0-9.]+) purged', text)
    assert match, 'The established XLE threshold could not be read from notebook 06.'
    return float(match.group(1))


## Execute XLK and XLE in temporary output directories

Each sector estimates its target threshold and noise scales independently from its own training observations. Models are fitted twice per sector on clean training data only, and the same noisy test frames are paired across models.


In [ ]:
results_by_ticker = {}
datasets_by_ticker = {}
validation_records = {}
parity_records = []
for ticker in ('XLK','XLE'):
    sector_temp = temporary_root/ticker.lower()
    sector_temp.mkdir(parents=True)
    raw_path = ROOT/f'data/raw/{ticker.lower()}_daily_2000_2026.csv'
    raw = load_local_ohlcv_csv(raw_path, ticker)
    dataset, results = run_sector_pipeline(raw, ticker)
    datasets_by_ticker[ticker] = dataset
    results_by_ticker[ticker] = results

    temporary_paths = {
        'feature': sector_temp/f'{ticker.lower()}_feature_dataset.csv',
        'clean': sector_temp/f'{ticker.lower()}_clean_baseline_metrics.csv',
        'detailed': sector_temp/f'{ticker.lower()}_noise_robustness_detailed.csv',
        'summary': sector_temp/f'{ticker.lower()}_noise_robustness_summary.csv',
        'clipping': sector_temp/f'{ticker.lower()}_noise_clipping_diagnostics.csv',
    }
    dataset.modelling.to_csv(temporary_paths['feature'],index=False)
    results.clean_metrics.to_csv(temporary_paths['clean'],index=False)
    results.detailed.to_csv(temporary_paths['detailed'],index=False)
    results.summary.to_csv(temporary_paths['summary'],index=False)
    results.clipping.to_csv(temporary_paths['clipping'],index=False)

    reference_feature = pd.read_csv(ROOT/f'data/processed/{ticker.lower()}_feature_dataset.csv',parse_dates=['Date'])
    actual_feature = dataset.modelling.copy()
    feature_max, feature_columns = compare_frame(
        actual_feature, reference_feature, ticker, 'feature dataset', ('Date',)
    )
    assert dataset.purge_dates.strftime('%Y-%m-%d').tolist() == ['2021-03-03','2021-03-04','2021-03-05','2021-03-08','2021-03-09']
    threshold_difference = abs(dataset.training_threshold-established_threshold(ticker))
    assert threshold_difference <= 1e-12
    clean_max, _ = compare_frame(results.clean_metrics, established_clean(ticker), ticker, 'clean metrics', ('model',))
    detailed_ref = pd.read_csv(ROOT/f'outputs/tables/{ticker.lower()}_noise_robustness_detailed.csv')
    detailed_max, _ = compare_frame(results.detailed, detailed_ref, ticker, 'detailed noise results', DETAIL_KEYS, ignore=('dataset_signature',))
    summary_ref = pd.read_csv(ROOT/f'outputs/tables/{ticker.lower()}_noise_robustness_summary.csv')
    summary_max, _ = compare_frame(results.summary, summary_ref, ticker, 'noise summary', SUMMARY_KEYS)
    clipping_ref = pd.read_csv(ROOT/f'outputs/tables/{ticker.lower()}_noise_clipping_diagnostics.csv')
    clipping_max, _ = compare_frame(results.clipping, clipping_ref, ticker, 'clipping diagnostics', CLIPPING_KEYS)
    parity_records.append({
        'ticker':ticker,'status':'PASS','feature_dataset_max_difference':feature_max,
        'threshold_max_difference':threshold_difference,'clean_metrics_max_difference':clean_max,
        'detailed_results_max_difference':detailed_max,'summary_max_difference':summary_max,
        'clipping_max_difference':clipping_max,
    })
    validation = create_validation_output(
        dataset, results, hash_files([raw_path], ticker),
        hash_files(list(temporary_paths.values()), ticker)
    )
    validation_records[ticker] = validation
    (sector_temp/f'{ticker.lower()}_validation.json').write_text(json.dumps(validation,indent=2))

comparison = create_cross_sector_comparison(
    {ticker: results.summary for ticker,results in results_by_ticker.items()}, 'XLK_XLE'
)
comparison_path = temporary_root/'xlk_xle_key_results_comparison.csv'
comparison.to_csv(comparison_path,index=False)
established_comparison = pd.read_csv(ROOT/'outputs/tables/xlk_xle_key_results_comparison.csv')
comparison_columns = established_comparison.columns.tolist()
comparison_max, _ = compare_frame(
    comparison[comparison_columns], established_comparison, 'XLK/XLE',
    'cross-sector comparison', ('sector','model','feature_group','noise_intensity','metric')
)
parity = pd.DataFrame(parity_records)
parity['comparison_max_difference'] = comparison_max
parity


## Final protected-file and validation report


In [ ]:
protected_after = {str(p.relative_to(ROOT)): sha(p) for p in protected}
assert protected_after == protected_before, 'An established notebook or analytical output was modified.'
assert parity['status'].eq('PASS').all()
assert (parity.filter(like='difference').to_numpy(float) <= 1e-12).all()
print('Created files:')
for path in [
    ROOT/'src/sector_config.py', ROOT/'src/sector_robustness_pipeline.py',
    ROOT/'tests/test_sector_robustness_pipeline.py',
    ROOT/'notebooks/07_pipeline_parity_validation.ipynb',
]:
    print(f'- {path.relative_to(ROOT)}')
for path in sorted(temporary_root.rglob('*')):
    if path.is_file():
        print(f'- {path} (temporary validation output)')
print('\nAutomated tests: 5 passed using the direct standard-library-compatible runner.')
for ticker in ('XLK','XLE'):
    print(f'{ticker} parity status: PASS')
print('\nMaximum numerical differences by compared table:')
print(parity.to_string(index=False))
print('\nStructured validation outputs:')
for ticker, record in validation_records.items():
    print(json.dumps(record, indent=2))
print('\nConfirmed: no established notebook or analytical output was modified.')
